In [1]:
!pip install pandas scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [2]:
!pip install gdown

In [3]:
import gdown

file_id = "1Cp-JkiAOfTS7VXv4H7In8kf6e8QKyz9R"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "historical_orders.csv",
    quiet=False
)

Downloading...
From: https://drive.google.com/uc?id=1Cp-JkiAOfTS7VXv4H7In8kf6e8QKyz9R
To: /content/historical_orders.csv
100%|██████████| 315k/315k [00:00<00:00, 56.2MB/s]


'historical_orders.csv'

In [4]:
import pandas as pd

df = pd.read_csv("historical_orders.csv")

print(df.shape)

df.head()

(4200, 11)


,order_id,lens_type,current_stage,order_age_hours,sla_hours,inventory_available,qc_failures,store_location,rework_count,delay_reason,breached
0,ORD-2024-100000,Single Vision,Frame Fitting,26.2,30,1,0,Chennai,0,Logistics Delay,1
1,ORD-2024-100001,Progressive,Prescription Verified,2.1,72,1,0,Bangalore,0,NaN,0
2,ORD-2024-100002,Single Vision,Frame Fitting,14.2,24,1,0,Bangalore,0,NaN,0
3,ORD-2024-100003,Single Vision,Shipped,21.9,24,1,0,Bangalore,0,NaN,0
4,ORD-2024-100004,Progressive,Shipped,56.4,60,1,0,Hyderabad,0,NaN,0


In [5]:
print(df.info())

print("\nClass Distribution:")
print(df["breached"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             4200 non-null   object 
 1   lens_type            4200 non-null   object 
 2   current_stage        4200 non-null   object 
 3   order_age_hours      4200 non-null   float64
 4   sla_hours            4200 non-null   int64  
 5   inventory_available  4200 non-null   int64  
 6   qc_failures          4200 non-null   int64  
 7   store_location       4200 non-null   object 
 8   rework_count         4200 non-null   int64  
 9   delay_reason         2372 non-null   object 
 10  breached             4200 non-null   int64  
dtypes: float64(1), int64(5), object(5)
memory usage: 361.1+ KB
None

Class Distribution:
breached
0    2979
1    1221
Name: count, dtype: int64


In [6]:
df = df.drop(columns=["order_id"])

df.head()

,lens_type,current_stage,order_age_hours,sla_hours,inventory_available,qc_failures,store_location,rework_count,delay_reason,breached
0,Single Vision,Frame Fitting,26.2,30,1,0,Chennai,0,Logistics Delay,1
1,Progressive,Prescription Verified,2.1,72,1,0,Bangalore,0,NaN,0
2,Single Vision,Frame Fitting,14.2,24,1,0,Bangalore,0,NaN,0
3,Single Vision,Shipped,21.9,24,1,0,Bangalore,0,NaN,0
4,Progressive,Shipped,56.4,60,1,0,Hyderabad,0,NaN,0


In [7]:
df["delay_reason"] = df["delay_reason"].fillna("None")

df.isnull().sum()

,0
lens_type,0
current_stage,0
order_age_hours,0
sla_hours,0
inventory_available,0
qc_failures,0
store_location,0
rework_count,0
delay_reason,0
breached,0


In [10]:
X = df.drop("breached", axis=1)

y = df["breached"]

print(X.shape)
print(y.shape)

(4200, 9)
(4200,)


In [12]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in X.select_dtypes(include="object").columns:
    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(X[col])

    encoders[col] = encoder

print(X.head())

   lens_type  current_stage  order_age_hours  sla_hours  inventory_available  \
0          2              1             26.2         30                    1   
1          1              5              2.1         72                    1   
2          2              1             14.2         24                    1   
3          2              7             21.9         24                    1   
4          1              7             56.4         60                    1   

   qc_failures  store_location  rework_count  delay_reason  
0            0               1             0             1  
1            0               0             0             3  
2            0               0             0             3  
3            0               0             0             3  
4            0               3             0             3  


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(3360, 9)
(840, 9)


In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

print("Training Completed")

Training Completed


In [17]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:,1]

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

Accuracy : 0.8738095238095238
Precision: 0.8
Recall   : 0.7540983606557377
F1 Score : 0.7763713080168776

Classification Report

              precision    recall  f1-score   support

           0       0.90      0.92      0.91       596
           1       0.80      0.75      0.78       244

    accuracy                           0.87       840
   macro avg       0.85      0.84      0.84       840
weighted avg       0.87      0.87      0.87       840



In [21]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df

,Feature,Importance
2,order_age_hours,0.496031
1,current_stage,0.098233
4,inventory_available,0.089508
8,delay_reason,0.071802
3,sla_hours,0.068225
5,qc_failures,0.063115
6,store_location,0.045437
7,rework_count,0.035146
0,lens_type,0.032503


In [23]:
import joblib

joblib.dump(model, "sla_model.pkl")

joblib.dump(encoders, "encoders.pkl")

print("Model Saved")

Model Saved


In [24]:
from google.colab import files

files.download("sla_model.pkl")
files.download("encoders.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance_df.sort_values(
    by="importance",
    ascending=False
)

,feature,importance
2,order_age_hours,0.496031
1,current_stage,0.098233
4,inventory_available,0.089508
8,delay_reason,0.071802
3,sla_hours,0.068225
5,qc_failures,0.063115
6,store_location,0.045437
7,rework_count,0.035146
0,lens_type,0.032503
